In [107]:
# !pip install requests pandas sentence-transformers hdbscan google-generativeai jupyter

In [108]:
# !pip install streamlit requests sentence-transformers hdbscan pandas numpy google-genai

In [109]:
# %pip install umap-learn

In [110]:
import umap
import hdbscan
from sentence_transformers import SentenceTransformer
import requests
import pandas as pd
import os
from google import genai
import numpy as np
from pydantic import BaseModel, Field
from google.genai.types import GenerateContentConfig
import requests

class TrendInsight(BaseModel):
    trend_name: str = Field(description="A catchy 2-to-4 word label for the trend.")
    key_ingredients_or_products: list[str] = Field(description="Specific products, ingredients, or tools explicitly mentioned in the posts.")
    consumer_pain_point: str = Field(description="The underlying problem or insecurity the consumers are trying to solve.")
    capitalization_strategy: str = Field(description="A 1-sentence idea on how a brand could capitalize on this specific trend. Make it directand actionable.")
    actionability_score: int = Field(description="A score from 1-10 on how easily a business could monetize this trend.")

class KeywordResponse(BaseModel):
    keywords: list[str] = Field(description="A list of semantically related words.")

# --- Credentials ---
BSKY_HANDLE = os.getenv("BSKY_HANDLE")
BSKY_APP_PASSWORD = os.getenv("BSKY_APP_PASSWORD")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")


In [ ]:
def get_related_keywords(primary_word: str, num_related: int = 4) -> list[str]:
    """Queries Gemini for social-media-optimized search terms related to `primary_word`."""
    client = genai.Client(api_key=GEMINI_API_KEY)
    
    prompt = f"""
    You are an expert social media analyst. 
    I am building a search query to find trending conversations about '{primary_word}' on platforms like Bluesky and Twitter.
    
    Generate exactly {num_related} highly relevant search terms that real users actually type in their posts. 
    Instead of formal dictionary synonyms, prioritize:
    - Popular slang or adjacent cultural trends
    - Niche communities
    - Highly associated routines, problems, or products
    
    Rules:
    - Keep terms short (1 to 2 words maximum).
    - Do NOT include the '#' symbol.
    """
    
    try:
        response = client.models.generate_content(
            model='gemini-3.5-flash-lite', 
            contents=prompt,
            config=GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=KeywordResponse,
            )
        )
        
        # Access the structured data safely using .parsed
        related_words = response.parsed.keywords
        
        # Ensure we only return the exact number requested 
        return related_words[:num_related]
        
    except Exception as e:
        print(f"Gemini API error: {e}")
        # Return an empty list so the app falls back to just the main keyword
        return []

In [112]:
def fetch_bluesky_posts(query, target_count):

    # 1. Create a session to get the auth token
    session_url = "https://bsky.social/xrpc/com.atproto.server.createSession"
    session_data = {"identifier": BSKY_HANDLE, "password": BSKY_APP_PASSWORD}
    session_resp = requests.post(session_url, json=session_data).json()
    
    if "accessJwt" not in session_resp:
        raise Exception(f"Failed to authenticate: {session_resp}")
        
    auth_token = session_resp["accessJwt"]
    headers = {"Authorization": f"Bearer {auth_token}"}
    
    # 2. Search for posts iteratively
    search_url = "https://bsky.social/xrpc/app.bsky.feed.searchPosts"
    
    posts_data = []
    cursor = None
    
    print(f"Fetching {target_count} posts for '{query}'...")
    while len(posts_data) < target_count:
        params = {"q": query, "limit": 100} # 100 is the max per request
        if cursor:
            params["cursor"] = cursor
            
        resp = requests.get(search_url, headers=headers, params=params).json()
        new_posts = resp.get("posts", [])
        
        if not new_posts:
            break # No more posts available
            
        for post in new_posts:
            # We extract the text, the timestamp, and the author
            posts_data.append({
                "text": post["record"]["text"],
                "created_at": post["record"]["createdAt"],
                "author": post["author"]["handle"],
                "replyCount": post.get("replyCount", 0),
                "repostCount": post.get("repostCount", 0),
                "likeCount": post.get("likeCount", 0),
                "quoteCount": post.get("quoteCount", 0)
                # "has_embed_link": has_embed,
                # "labels": labels
            })
            
        cursor = resp.get("cursor")
        if not cursor:
            break
            
    # Keep only the target amount and convert to a DataFrame
    df = pd.DataFrame(posts_data[:target_count])
    print(f"Successfully fetched {len(df)} posts.")
    return df

# Test
# df_posts = fetch_bluesky_posts("skincare", target_count=5000)
# df_posts.head()

In [113]:
def filter_spam_posts(df, threshold):
    """
    Calculates a spam score (0.0 to 1.0) based on engagement, duplication, 
    and text formatting, then filters out posts above the threshold.
    """
    # Create a copy to avoid SettingWithCopyWarning
    df_scored = df.copy()
    
    # Initialize score
    df_scored['spam_score'] = 0.0
    
    # 1. Duplication Penalty (Strongest signal)
    # Flag posts that share the exact same text (e.g., cross-posting bots)
    is_duplicate = df_scored.duplicated(subset=['text'], keep='first')
    df_scored.loc[is_duplicate, 'spam_score'] += 0.5
    
    # 2. Low Engagement Penalty
    # Summing up the engagement metrics you are already fetching
    df_scored['total_engagement'] = (
        df_scored['replyCount'] + 
        df_scored['repostCount'] + 
        df_scored['likeCount'] + 
        df_scored['quoteCount']
    )
    # Add a penalty if the post has completely zero engagement
    df_scored.loc[df_scored['total_engagement'] == 0, 'spam_score'] += 0.2
    
    # 3. Content Heuristics (Links & Hashtags)
    # Count occurrences using basic regex
    df_scored['hashtag_count'] = df_scored['text'].str.count(r'#\w+')
    df_scored['link_count'] = df_scored['text'].str.count(r'http[s]?://')
    
    # Penalize spammy text formatting
    df_scored.loc[df_scored['hashtag_count'] > 4, 'spam_score'] += 0.15
    df_scored.loc[df_scored['link_count'] >= 2, 'spam_score'] += 0.15
    
    # 4. Cap the maximum score at 1.0
    df_scored['spam_score'] = df_scored['spam_score'].clip(upper=1.0)
    
    # Filter the DataFrame based on the acceptable threshold
    initial_count = len(df_scored)
    df_filtered = df_scored[df_scored['spam_score'] < threshold].copy()
    filtered_count = len(df_filtered)
    
    print(f"Filtered out {initial_count - filtered_count} spam-likely posts.")
    
    # Clean up calculation columns before passing to the clustering phase
    df_filtered = df_filtered.drop(columns=['total_engagement', 'hashtag_count', 'link_count'])
    
    return df_filtered


# df_posts = fetch_bluesky_posts("skincare", target_count=5000)
# df_clean = filter_spam_posts(df_posts, threshold=0.6)
# df_clustered = cluster_social_posts(df_clean)

In [114]:
# df_clean

In [115]:
def cluster_social_posts(df, cluster_fraction, sample_fraction):
    print("Loading Sentence Transformer model...")
    model = SentenceTransformer('all-MiniLM-L6-v2') 
    
    print("Generating embeddings...")
    embeddings = model.encode(df['text'].tolist())
    
    print("Reducing dimensions with UMAP...")
    # Compress the 384 dimensions down to 5 to help HDBSCAN find density
    umap_model = umap.UMAP(
        n_neighbors=30, # Focuses on local neighborhood size 
        n_components=5, # Reduce to 5 dimensions
        min_dist=0.0,   # How tightly to pack points together 
        metric='cosine',# Cosine works best for text embeddings
        random_state=42 # Ensure reproducible results
    )
    reduced_embeddings = umap_model.fit_transform(embeddings)
    
    print("Running HDBSCAN clustering...")
    
    # Calculate dynamic parameters based on DataFrame size
    total_posts = len(df)
    
    # Force the values to be integers, and set an absolute minimum floor (e.g., 5)
    # so small datasets don't end up with a min_cluster_size of 1.
    dynamic_min_cluster_size = max(5, int(total_posts * cluster_fraction))
    dynamic_min_samples = max(5, int(dynamic_min_cluster_size * 0.5))
    
    print(f"Dynamic Settings: min_cluster_size={dynamic_min_cluster_size}, min_samples={dynamic_min_samples}")
    
    print("Running HDBSCAN clustering...")
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=dynamic_min_cluster_size, 
        min_samples=dynamic_min_samples,      
        metric='euclidean'
        # , cluster_selection_epsilon=0.05  
    )


    df['cluster_id'] = clusterer.fit_predict(reduced_embeddings)
    
    # -1 means "noise" (unclustered). Let's filter those out.
    clustered_df = df[df['cluster_id'] != -1]
    
    print(f"Found {len(clustered_df['cluster_id'].unique())} unique clusters.")
    return clustered_df
# Test
# df_clustered = cluster_social_posts(df_clean, 0.01 , 0.002)
# print(df_clustered['cluster_id'].value_counts())

In [116]:
# # Test keywords

# # lst_keywords = ["skincare", "toothpaste", "haircare", "electric vehicle", "mens fashion", "stocks", "investment", "fitness"]

# lst_keywords = ["music"]

# for keyword in lst_keywords:
#     print(keyword)
#     df_posts = fetch_bluesky_posts(keyword, target_count=5000)
#     df_clean = filter_spam_posts(df_posts, threshold=0.1)
#     df_clustered = cluster_social_posts(df_clean, 0.01 , 0.002)
#     print(df_clustered['cluster_id'].value_counts())
#     # df_clustered = pd.DataFrame()



In [117]:
def extract_actionable_insights(df_clustered):
    client = genai.Client(api_key=GEMINI_API_KEY)
    
    # Create a dictionary to hold our rich insights
    cluster_insights = {}
    # Count the posts in each cluster and sort them from largest to smallest
    cluster_sizes = df_clustered['cluster_id'].value_counts()
    
    # Grab the IDs of the top 6 largest clusters
    top_6_clusters = cluster_sizes.head(6).index.tolist()
    
# Iterate ONLY over those top 6
    for cluster_id in top_6_clusters:
        
        # Isolate the dataframe to just the posts in the current cluster
        cluster_df = df_clustered[df_clustered['cluster_id'] == cluster_id].copy()
        
        # Calculate total interactions for these specific posts
        cluster_df['total_interactions'] = (
            cluster_df['likeCount'] + 
            cluster_df['repostCount'] + 
            cluster_df['replyCount'] + 
            cluster_df['quoteCount']
        )
        
        # Sort by the most interacted posts first, then grab the top 15
        sample_posts = cluster_df.sort_values(by='total_interactions', ascending=False)['text'].head(15).tolist()

        posts_text = "\n- ".join(sample_posts)
        
        prompt = f"""
        You are an expert consumer trend analyst and product developer. 
        Analyze the following social media posts that have been clustered together:
        - {posts_text}
        
        Extract the underlying trend and identify exactly how a business can capitalize on it.
        """
        
        # Enforce structured output via GenerateContentConfig
        response = client.models.generate_content(
            model='gemini-3.5-flash-lite', 
            contents=prompt,
            config=GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=TrendInsight,
            )
        )
        
        # Access the structured data safely using .parsed
        insight = response.parsed
        cluster_insights[cluster_id] = insight
        
        print(f"Analyzed Cluster {cluster_id}: {insight.trend_name}. \nProduct: {insight.capitalization_strategy} (Score: {insight.actionability_score}/10)")
        
    # Map the new structured data back to the dataframe
    df_clustered['trend_name'] = df_clustered['cluster_id'].map(lambda x: cluster_insights[x].trend_name if x in cluster_insights else None)
    df_clustered['key_products'] = df_clustered['cluster_id'].map(lambda x: ", ".join(cluster_insights[x].key_ingredients_or_products) if x in cluster_insights else None)
    df_clustered['pain_point'] = df_clustered['cluster_id'].map(lambda x: cluster_insights[x].consumer_pain_point if x in cluster_insights else None)
    df_clustered['strategy'] = df_clustered['cluster_id'].map(lambda x: cluster_insights[x].capitalization_strategy if x in cluster_insights else None)
    df_clustered['actionability_score'] = df_clustered['cluster_id'].map(lambda x: cluster_insights[x].actionability_score if x in cluster_insights else None)
    
    return df_clustered

In [120]:
# 1. Define seed keyword and settings
seed_keyword = "skincare"
posts_per_keyword = 1500 

# 2. Fetch related keywords
print(f"Fetching related keywords for '{seed_keyword}'...")
related_words = get_related_keywords(seed_keyword, num_related = 3)

# 3. Combine the seed keyword with the related words
all_keywords = [seed_keyword] + related_words
all_keywords = [item.replace(" ", "") for item in all_keywords]
print(f"Expanded search keywords: {all_keywords}")

# 4. Fetch posts for all keywords
all_posts = []
for keyword in all_keywords:
    print(f"\n--- Fetching data for: {keyword} ---")
    df_temp = fetch_bluesky_posts(keyword, target_count=posts_per_keyword)
    all_posts.append(df_temp)

# 5. Combine everything into one master DataFrame
df_combined_posts = pd.concat(all_posts, ignore_index=True)
print(f"\nTotal posts collected across all keywords: {len(df_combined_posts)}")

# 6. Run the rest of your pipeline on the combined dataset
print("\n--- Filtering Spam ---")
df_clean = filter_spam_posts(df_combined_posts, threshold = 0.1)

print("\n--- Clustering Posts ---")
df_clustered = cluster_social_posts(df_clean, cluster_fraction = 0.01, sample_fraction = 0.002)



Fetching related keywords for 'skincare'...
Expanded search keywords: ['skincare', 'glassskin', 'slugging', 'damagedbarrier']

--- Fetching data for: skincare ---
Fetching 1500 posts for 'skincare'...
Successfully fetched 1500 posts.

--- Fetching data for: glassskin ---
Fetching 1500 posts for 'glassskin'...
Successfully fetched 290 posts.

--- Fetching data for: slugging ---
Fetching 1500 posts for 'slugging'...
Successfully fetched 1500 posts.

--- Fetching data for: damagedbarrier ---
Fetching 1500 posts for 'damagedbarrier'...
Successfully fetched 0 posts.

Total posts collected across all keywords: 3290

--- Filtering Spam ---
Filtered out 1725 spam-likely posts.

--- Clustering Posts ---
Loading Sentence Transformer model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Generating embeddings...
Reducing dimensions with UMAP...


c:\Users\JesusSantillanMinila\Documents\GitHub\social-media-trend-detector\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Running HDBSCAN clustering...
Dynamic Settings: min_cluster_size=15, min_samples=7
Running HDBSCAN clustering...
Found 3 unique clusters.


In [121]:
print("\n--- Extracting Actionable Insights ---")
df_labeled = extract_actionable_insights(df_clustered)

# View the final structured output
df_labeled


--- Extracting Actionable Insights ---
Analyzed Cluster 2: Skincare Slugging Routine. 
Product: Launch a targeted nighttime occlusion balm marketed specifically as a lightweight alternative to traditional heavy slugging products. (Score: 8/10)
Analyzed Cluster 1: No-Nonsense Skincare. 
Product: Brands can capitalize on this trend by launching streamlined, all-in-one starter kits that emphasize affordability, transparency, and essential steps like hydration and sun protection. (Score: 9/10)
Analyzed Cluster 0: Skincare Fatigue and Skepticism. 
Product: Launch a minimalist, simplified skincare line with transparent, clinically-backed single-step products and educational marketing that cuts through the noise. (Score: 8/10)


,text,created_at,author,replyCount,repostCount,likeCount,quoteCount,spam_score,cluster_id,trend_name,key_products,pain_point,strategy,actionability_score
1,And guess what. \n\nNot going back. \n\nNot on...,2026-08-03T18:14:31.829Z,pbolton.bsky.social,1,1,1,0,0.0,1,No-Nonsense Skincare,"sunscreen, neutral moisturizing lotion, eye pa...",Consumers are experiencing fatigue from overly...,Brands can capitalize on this trend by launchi...,9
2,This is how we get white men to do skincare,2026-08-03T18:09:02.122Z,maladroithe.bsky.social,0,0,17,0,0.0,1,No-Nonsense Skincare,"sunscreen, neutral moisturizing lotion, eye pa...",Consumers are experiencing fatigue from overly...,Brands can capitalize on this trend by launchi...,9
9,Self-care is not something you have to earn or...,2026-08-03T17:30:26.709Z,aloebud.bsky.social,0,1,2,0,0.0,1,No-Nonsense Skincare,"sunscreen, neutral moisturizing lotion, eye pa...",Consumers are experiencing fatigue from overly...,Brands can capitalize on this trend by launchi...,9
12,📖 In UT: Natural History Museum of Utah. The S...,2026-08-03T17:16:40.855935+00:00,travelingadvisor.bsky.social,1,0,2,0,0.0,1,No-Nonsense Skincare,"sunscreen, neutral moisturizing lotion, eye pa...",Consumers are experiencing fatigue from overly...,Brands can capitalize on this trend by launchi...,9
14,A Comprehensive Guide to Glossier Pregnancy-Sa...,2026-08-03T17:02:58+00:00,healthandfitness2.bsky.social,0,0,1,0,0.0,1,No-Nonsense Skincare,"sunscreen, neutral moisturizing lotion, eye pa...",Consumers are experiencing fatigue from overly...,Brands can capitalize on this trend by launchi...,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3281,👮Have you seen this Werewolf Vtuber? Under Arr...,2026-04-21T17:44:42.301Z,dcypherpup.bsky.social,1,0,3,0,0.0,2,Skincare Slugging Routine,"Vaseline, Aquaphor, Botox, face tape","Dry, aging, or damaged skin barriers that need...",Launch a targeted nighttime occlusion balm mar...,8
3286,"Turang’s slugging .540 over his last, like, 38...",2026-04-21T13:21:20.146Z,matrueblood.bsky.social,1,0,1,0,0.0,2,Skincare Slugging Routine,"Vaseline, Aquaphor, Botox, face tape","Dry, aging, or damaged skin barriers that need...",Launch a targeted nighttime occlusion balm mar...,8
3287,🔴LIVE SOON!!\nIt's slugging time :D\n- www.you...,2026-04-21T11:58:45.725Z,linacynth.bsky.social,0,0,1,0,0.0,2,Skincare Slugging Routine,"Vaseline, Aquaphor, Botox, face tape","Dry, aging, or damaged skin barriers that need...",Launch a targeted nighttime occlusion balm mar...,8
3288,Many of us SABR types believed that Choi's pow...,2026-04-21T03:36:29.685Z,hunterfelt.bsky.social,1,1,5,0,0.0,2,Skincare Slugging Routine,"Vaseline, Aquaphor, Botox, face tape","Dry, aging, or damaged skin barriers that need...",Launch a targeted nighttime occlusion balm mar...,8
